# Matemáticas de la Inteligencia Artificial
## Sesión 9 — Grafos: aprender cuando lo importante son las relaciones

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CuentosCuanticos/matematicas-ia/blob/main/09_grafos_gnn/laboratorio.ipynb)

### Pregunta de la sesión
**¿Cómo hacemos que una máquina utilice no solo las características de cada objeto, sino también las relaciones entre objetos?**

Recorrido: $\text{grafo}\to A,D,B,L\to\text{espectro}\to\text{difusión}\to\text{message passing}\to\text{GCN}\to\text{atención}$.

Completa los bloques `TODO`. NetworkX se utiliza para visualizar; el álgebra del grafo se construye explícitamente con NumPy. PyTorch aparece solo al final para optimizar un clasificador lineal ya comprendido en sesiones anteriores.

## Diccionario matemática–código

| Matemática | Código | Significado |
|---|---|---|
| $G=(V,E)$ | `G` | grafo |
| $A$ | `A` | adyacencia |
| $D$ | `D` | matriz de grados |
| $B$ | `B` | incidencia con signo |
| $L=D-A$ | `L` | laplaciano |
| $X$ | `X` | características nodales |
| $H^{(\ell)}$ | `H` | representación de una capa |
| $W^{(\ell)}$ | `W` | parámetros aprendibles |
| $\alpha_{ij}$ | `alpha[i,j]` | peso de atención de $j$ hacia $i$ |

En matemáticas numeramos los nodos $1,2,3,4$; en Python usaremos índices `0,1,2,3`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
np.set_printoptions(precision=4,suppress=True)
rng=np.random.default_rng(9)

# 1. Grafo, adyacencia, grados y laplaciano

Usamos $E=\{\{1,2\},\{1,3\},\{2,3\},\{3,4\}\}$. Construye $A$, luego $D=\operatorname{diag}(d_i)$ y $L=D-A$. Verifica que $L\mathbf1=0$.

In [ ]:
n=4
edges=[(0,1),(0,2),(1,2),(2,3)]
G=nx.Graph(); G.add_nodes_from(range(n)); G.add_edges_from(edges)
pos={0:(0,1),1:(1,1),2:(.5,.2),3:(.5,-.8)}
nx.draw(G,pos=pos,labels={i:str(i+1) for i in range(n)},with_labels=True,node_size=900)
plt.show()

def matriz_adyacencia(n,edges):
    A=np.zeros((n,n),dtype=float)
    for i,j in edges:
        # TODO
        ...
    return A

A=matriz_adyacencia(n,edges)
# TODO
grados=...
D=...
L=...
print('A=\n',A)
print('grados=',grados)
print('L=\n',L)
print('sumas de filas de L=',L.sum(axis=1))

# 2. Señales, incidencia y energía

Para $\mathbf f=(1,0,1,1)^T$, calcula $A\mathbf f$ y $L\mathbf f$. Después construye la incidencia orientada auxiliar $B$ para $1\to2$, $1\to3$, $2\to3$, $3\to4$. Comprueba

$$\mathcal E(\mathbf f)=\frac12\mathbf f^TL\mathbf f=\frac12\|B^T\mathbf f\|_2^2.$$

In [ ]:
f=np.array([1.,0.,1.,1.])
Af=... # TODO
Lf=... # TODO

orientadas=[(0,1),(0,2),(1,2),(2,3)]
B=np.zeros((n,len(orientadas)))
for e,(i,j) in enumerate(orientadas):
    # TODO
    ...

dif=B.T@f
E1=... # TODO
E2=... # TODO
print('Af=',Af,'Lf=',Lf)
print('B^T f=',dif)
print('energías=',E1,E2)

# 3. Espectro y difusión

Diagonaliza $L$. En un grafo conexo, el cero tiene multiplicidad uno y $\lambda_2>0$. Después implementa Euler explícito

$$\mathbf f^{(k+1)}=(I-\tau L)\mathbf f^{(k)}.$$

Sigue la suma $\sum_i f_i$ y la energía $\frac12\mathbf f^TL\mathbf f$. Prueba también un $\tau$ superior a $2/\lambda_{\max}$.

In [ ]:
autovalores,autovectores=... # TODO
print('autovalores=',autovalores)
print('lambda2=',autovalores[1])

def energia(f,L):
    return ... # TODO

def difundir(f0,L,tau=.25,pasos=16):
    f=f0.astype(float).copy(); hist=[f.copy()]; Es=[energia(f,L)]
    for _ in range(pasos):
        f=... # TODO
        hist.append(f.copy()); Es.append(energia(f,L))
    return np.array(hist),np.array(Es)

hist,Es=difundir(f,L,.25,16)
print('suma inicial/final=',hist[0].sum(),hist[-1].sum())
for i in range(n): plt.plot(hist[:,i],marker='o',ms=3,label=f'nodo {i+1}')
plt.axhline(f.mean(),ls='--'); plt.legend(); plt.show()
plt.plot(Es,marker='o'); plt.show()

# 4. Caminata aleatoria y equivarianza

Construye $P=D^{-1}A$. Recuerda que $P^T\mathbf p$ transporta una distribución y $P\mathbf g$ promedia una señal. Verifica también la distribución estacionaria $\pi_i=d_i/\sum_jd_j$.

Después permuta los nodos y comprueba $A'X'=P_\pi AX$ con $A'=P_\pi AP_\pi^T$ y $X'=P_\pi X$.

In [ ]:
P_rw=... # TODO
pi=... # TODO
print('error estacionario=',np.linalg.norm(P_rw.T@pi-pi))

X=np.array([[1.,0.],[0.,1.],[1.,1.],[1.,0.]])
orden=np.array([2,0,3,1]); P_perm=np.eye(n)[orden]
A_perm=... # TODO
X_perm=... # TODO
print('error equivarianza=',np.linalg.norm(A_perm@X_perm-P_perm@(A@X)))

# 5. Message passing y GCN

Implementa

$$H'=\sigma\left(HW_{\rm self}^T+(AH)W_{\rm neigh}^T+b\right)$$

y después la normalización GCN

$$\widehat A=\widetilde D^{-1/2}\widetilde A\widetilde D^{-1/2},\qquad H'=\sigma(\widehat AHW^T).$$

Observa especialmente qué ocurre con los nodos 1 y 4: parten de atributos idénticos pero tienen entornos diferentes.

In [ ]:
relu=lambda Z: np.maximum(Z,0)
W_self=np.array([[1.,-.2],[.3,.8]])
W_neigh=np.array([[.5,.4],[-.1,.6]])
b=np.array([.05,-.05])

def message_passing(A,H,W_self,W_neigh,b):
    agregado=... # TODO
    pre=... # TODO
    return relu(pre)

H1=message_passing(A,X,W_self,W_neigh,b)
print('H1=\n',H1)

def operador_gcn(A):
    At=A+np.eye(A.shape[0]); dt=At.sum(axis=1)
    Dm12=... # TODO
    return ... # TODO

A_hat=operador_gcn(A)
W=np.array([[1.,-.4],[.2,.8]])
H_gcn=... # TODO
print('A_hat=\n',A_hat)
print('H_gcn=\n',H_gcn)

# 6. Un experimento mínimo de aprendizaje

Las etiquetas serán $y=(0,0,0,1)$. Los nodos 1 y 4 tienen el mismo vector de entrada, de modo que un clasificador fila a fila no puede distinguirlos. Compara el mismo clasificador lineal entrenado sobre $X$ y sobre $\widehat AX$.

In [ ]:
import torch
import torch.nn as nn
torch.manual_seed(9)
y=np.array([0,0,0,1]); X_grafo=A_hat@X

def entrenar_lineal(Xin,y,epocas=400,lr=.1):
    Xt=torch.tensor(Xin,dtype=torch.float32); yt=torch.tensor(y,dtype=torch.long)
    modelo=nn.Linear(Xin.shape[1],2); optim=torch.optim.Adam(modelo.parameters(),lr=lr); loss_fn=nn.CrossEntropyLoss()
    for _ in range(epocas):
        optim.zero_grad(); logits=modelo(Xt); loss=loss_fn(logits,yt); loss.backward(); optim.step()
    with torch.no_grad(): pred=modelo(Xt).argmax(1).numpy()
    return pred

pred_raw=... # TODO
pred_grafo=... # TODO
print('raw=',pred_raw,'accuracy=',np.mean(pred_raw==y))
print('grafo=',pred_grafo,'accuracy=',np.mean(pred_grafo==y))

# 7. Atención: aprender a quién escuchar

Usaremos una GAT elemental:

$$z_i=Wh_i,\quad e_{ij}=\operatorname{LeakyReLU}(a^T[z_i\Vert z_j]),\quad \alpha_{ij}=\operatorname{softmax}_j(e_{ij}),\quad h_i'=\sum_j\alpha_{ij}z_j.$$

La adyacencia decide quién puede hablar; la atención decide cuánto se escucha.

In [ ]:
def leaky_relu(x,p=.2): return np.where(x>=0,x,p*x)

def atencion_grafo(A,H,W,a):
    n=A.shape[0]; At=A+np.eye(n); Z=H@W.T; scores=np.full((n,n),-np.inf)
    for i in range(n):
        for j in np.where(At[i]>0)[0]:
            par=np.concatenate([Z[i],Z[j]])
            scores[i,j]=... # TODO
    alpha=np.zeros((n,n))
    for i in range(n):
        js=np.where(At[i]>0)[0]; s=scores[i,js]
        exps=... # TODO, softmax estable
        alpha[i,js]=... # TODO
    return ... ,alpha,scores # TODO

W_att=np.array([[1.,.2],[-.3,.9]]); a_att=np.array([.8,-.4,.5,1.])
H_att,alpha,scores=atencion_grafo(A,X,W_att,a_att)
print('alpha=\n',alpha)
print('sumas=',alpha.sum(axis=1))
plt.imshow(alpha); plt.colorbar(); plt.xlabel('emisor j'); plt.ylabel('receptor i'); plt.show()

# Problema final abierto — Diseñar una propagación robusta

Hay 10 nodos, dos regiones densas, tres nodos sin etiqueta y un nodo con atributos poco fiables. Las dos primeras columnas de `X_reto` son características; la tercera es fiabilidad. En `y_parcial`, `-1` significa etiqueta desconocida.

### Requisitos
1. Construye $A,D,L$ y estudia $\lambda_2$ y el vector de Fiedler.
2. Diseña una primera estrategia basada en difusión o promedio vecinal.
3. Diseña una segunda estrategia donde la fiabilidad modifique los pesos, por ejemplo $\alpha_{ij}\propto e^{\gamma r_j}$.
4. Compara al menos dos reglas de propagación y estudia un hiperparámetro.
5. Permuta los nodos y verifica que, al deshacer la permutación, obtienes la misma respuesta.
6. Predice las etiquetas desconocidas y redacta 8–12 líneas justificando tu decisión.

No hay una única solución correcta. La versión docente contiene una solución de referencia.

In [ ]:
n_reto=10
edges_reto=[(0,1),(0,2),(1,2),(1,3),(2,3),(3,4),(5,6),(5,7),(6,7),(6,8),(7,8),(8,9),(4,5),(2,7)]
X_reto=np.array([[1.,.1,.95],[.9,.2,.90],[1.1,0,.95],[.8,.25,.85],[.45,.55,.70],[.55,.45,.80],[.15,.9,.95],[.75,.25,.25],[.1,1.,.90],[.2,.85,.90]])
y_parcial=np.array([0,0,0,0,-1,-1,1,-1,1,1])
G_reto=nx.Graph(); G_reto.add_nodes_from(range(n_reto)); G_reto.add_edges_from(edges_reto)
pos_reto=nx.spring_layout(G_reto,seed=9)
nx.draw(G_reto,pos_reto,labels={i:str(i+1) for i in range(n_reto)},with_labels=True,node_size=800)
plt.show()
print('sin etiqueta:',np.where(y_parcial<0)[0]+1)
print(X_reto)

In [ ]:
# RETO A: diagnóstico estructural
A_r=...; D_r=...; L_r=...; evals_r,evecs_r=...

# RETO B: primera estrategia
# TODO

# RETO C: segunda estrategia ponderada/atención
# TODO

# RETO D: robustez y prueba de permutación
# TODO

# RETO E: escribe aquí tu conclusión final
